# SVM Model - PharmShed Super Dataset

**Model:** Linear SVM (LinearSVC, sklearn)
**Task:** Multi-class classification — predict which of 217 pharmaceuticals a person is prescribed
**Features:** Demographics (Age, Sex, Family_income, Insurance_coverage, Race_ethnicity) + Prescription (Quantity, Form, Strength, Day_Supply)
**Split strategy:** StratifiedGroupKFold (5-fold CV), grouped by Person_ID to prevent data leakage
**Missing values:** Prescription NaNs (no-prescription rows) filled with -1 sentinel; demographic NaNs imputed with train-fold median inside CV loop
**Preprocessing:** Categorical features one-hot encoded; all numeric features standardized (StandardScaler) — SVM is margin-based and requires scaling
**Metrics:** Accuracy, Cohen's Kappa, MCC, macro/micro averaged Precision, Recall, F1, F2
**Output:** Saves `svm_super_proba_2022.csv` — probability vector over 217 drugs per observation, used as input to the ensemble model

## Key Design Decisions

**Why LinearSVC instead of SVC?**
SVC with a kernel (e.g. RBF) scales as O(n²) to O(n³) with training size — completely impractical for 900k+ rows.
LinearSVC uses a much more efficient solver (liblinear) that scales linearly with data size and number of features.
For high-dimensional data after one-hot encoding, linear kernels are also often competitive with RBF.

**Why -1 for prescription NaNs?**
The 97,497 'no prescriptions' rows have NaN for Quantity/Form/Strength/Day_Supply because those fields are structurally absent — not unknown.
Filling with -1 preserves this as a meaningful signal. After StandardScaler, -1 becomes a distinct region in feature space away from real prescription values.

**Why median impute demographic NaNs inside the fold?**
Age and Family_income have some missingness. Imputing inside each fold (on train split only) prevents leakage from val/test data influencing the imputation.

**Why StandardScaler?**
SVM maximizes the margin — unscaled features would make the margin geometry meaningless.
Features with large ranges (e.g. Family_income ~0–100k) would dominate the decision boundary without scaling.

**Why one-hot encode categoricals?**
LinearSVC operates in a dot-product space — ordinal integer codes for unordered categories would imply false ordinal distances. One-hot encoding treats each category as an independent binary dimension.

**Why class_weight='balanced'?**
The 217-class problem is severely imbalanced (e.g. atorvastatin: 38k rows vs ivermectin: 29 rows).
Balanced weighting automatically scales each class's penalty by inverse frequency, giving rare drugs a proportionally larger influence on the margin.

**Why CalibratedClassifierCV for probabilities?**
LinearSVC does not natively output probability estimates — it only outputs decision function scores.
CalibratedClassifierCV wraps LinearSVC and uses Platt scaling (sigmoid calibration) to convert
decision scores into proper probabilities over all 217 classes.
This is required for soft voting in the ensemble — all 5 models must output probability vectors.

In [ ]:
# Install required libraries.
# scikit-learn: LinearSVC model, preprocessing, CV, metrics.
# permetrics: macro/micro F2 score and other evaluation metrics.
!pip install scikit-learn permetrics

In [ ]:
# Import all required libraries.
import pandas as pd
import numpy as np
import sklearn
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from sklearn.utils import shuffle
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

# Confirm versions for reproducibility
print('scikit-learn version:', sklearn.__version__)
print('pandas version:', pd.__version__)
print('numpy version:', np.__version__)

# OneHotEncoder parameter name changed from 'sparse' to 'sparse_output' in sklearn 1.2.
# This flag ensures the notebook runs on both older and newer sklearn versions.
from packaging import version
OHE_SPARSE_KWARG = 'sparse_output' if version.parse(sklearn.__version__) >= version.parse('1.2') else 'sparse'

# CalibratedClassifierCV note:
# LinearSVC does not have predict_proba() — it only outputs decision function scores.
# Wrapping with CalibratedClassifierCV applies Platt scaling (sigmoid calibration)
# to convert decision scores into proper probability estimates over all 217 classes.
# This is required for the ensemble — all 5 base models must output probability vectors.
# cv='prefit' is used when the model is already trained — calibration is applied after fitting.
print('CalibratedClassifierCV imported for probability calibration.')

In [ ]:
# Load the super integrated dataset (2014-2021) and metadata.
# The super dataset contains demographics + prescription features joined on Observation_ID.
# Metadata provides Person_ID which is used only for StratifiedGroupKFold grouping.

DATA_DIR = './'  # change this to your data path if needed

super_df = pd.read_csv(f'{DATA_DIR}super_integrated_data.csv')
metadata = pd.read_csv(f'{DATA_DIR}metadata.csv')

if 'Unnamed: 0' in super_df.columns:
    super_df = super_df.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in metadata.columns:
    metadata = metadata.drop(columns=['Unnamed: 0'])

print('Super dataset shape:', super_df.shape)
print('Metadata shape:', metadata.shape)
print('\nSuper dataset columns:', super_df.columns.tolist())
print('\nMissing values:')
print(super_df.isnull().sum())

# Sanity check — every Observation_ID in super_df must exist in metadata
# so the Person_ID join in the next cell is complete with no dropped rows.
missing_ids = set(super_df['Observation_ID']) - set(metadata['Observation_ID'])
print(f'\nObservation_IDs in super_df missing from metadata: {len(missing_ids)}')

In [ ]:
# Join Person_ID from metadata onto the super dataset.
# Person_ID is not a model feature — it is used only to group records by person
# in StratifiedGroupKFold so that all rows for the same person stay in the same fold.
# This prevents data leakage from the same person appearing in both train and validation.
person_id_map = metadata[['Observation_ID', 'Person_ID']]
super_df = super_df.merge(person_id_map, on='Observation_ID', how='left')

print('Columns after join:', super_df.columns.tolist())
print('Shape after join:', super_df.shape)
print('Missing Person_IDs:', super_df['Person_ID'].isnull().sum())

# Hard stop if any Person_IDs are missing after the join.
# A missing Person_ID means that row has no group assignment for StratifiedGroupKFold,
# which would cause the CV split to fail or silently assign wrong groups.
assert super_df['Person_ID'].isnull().sum() == 0, \
    "ERROR: Missing Person_IDs after join — check metadata coverage before proceeding."

print('All Person_IDs joined successfully.')
print('Unique persons:', super_df['Person_ID'].nunique())

In [ ]:
# Prescription NaN handling — fill with -1 sentinel.
# The 97,497 no-prescription rows have NaN for Quantity/Strength/Day_Supply/Form
# because those fields are structurally absent (not missing at random).
# Filling with -1 preserves this as a meaningful signal distinct from real values.
# After StandardScaler, -1 maps to a distinct region in feature space.
#
# Demographic NaNs (Age, Family_income) are handled inside the CV loop
# using median imputation on the train split only — see the CV cell below.

numeric_prescription_cols = ['Quantity', 'Strength', 'Day_Supply']
super_df[numeric_prescription_cols] = super_df[numeric_prescription_cols].fillna(-1)

# Form is a categorical string column (values like TABS, ORAL, CAPS)
# Fill with string '-1' so it is treated as a distinct 'no prescription' category
# by the OneHotEncoder later.
super_df['Form'] = super_df['Form'].fillna('-1')

print('Missing values after prescription fill:')
print(super_df[['Quantity', 'Form', 'Strength', 'Day_Supply']].isnull().sum())
print('\nRemaining demographic NaNs (will be imputed inside CV loop):')
print(super_df[['Age', 'Family_income']].isnull().sum())

# Hard stop if prescription columns still have NaNs after fill.
# Unlike XGBoost, LinearSVC cannot handle NaN values natively.
# Any remaining NaN would corrupt the StandardScaler and cause the CV loop to crash.
assert super_df[numeric_prescription_cols + ['Form']].isnull().sum().sum() == 0, \
    "ERROR: Prescription NaNs remain after fill — check fillna logic before proceeding."

print('\nAll prescription NaNs filled successfully.')

In [ ]:
# Exploratory check before modeling.
# Verify class count, drug distribution, and person count.
print('Unique persons:', super_df['Person_ID'].nunique())
print('Unique drugs:', super_df['Drug'].nunique())
print('No prescription rows:', (super_df['Drug'] == 'no prescriptions').sum())
print('Actual drug rows:', (super_df['Drug'] != 'no prescriptions').sum())

print('\nTop 10 most prescribed drugs:')
print(super_df['Drug'].value_counts().head(10))

print('\nBottom 5 rarest drugs:')
print(super_df['Drug'].value_counts().tail(5))

# Check class imbalance ratio — max count / min count.
# High ratio confirms severe imbalance and justifies class_weight='balanced'
# in LinearSVC — rare drugs need proportionally higher penalty to be learned.
counts = super_df['Drug'].value_counts()
print(f'\nMost common drug count:  {counts.max():,}')
print(f'Rarest drug count:       {counts.min():,}')
print(f'Imbalance ratio:         {counts.max() / counts.min():.1f}x')

# Hard stop if drug count is not 217.
# Any deviation means the dataset changed or a merge dropped/added classes
# and the LabelEncoder in the next cell would produce wrong mappings.
assert super_df['Drug'].nunique() == 217, \
    f"ERROR: Expected 217 drug classes, found {super_df['Drug'].nunique()}."

print('\nDrug class count confirmed: 217.')

In [ ]:
# Define feature columns, categorical columns, numeric columns, and target.
#
# Categorical features are one-hot encoded (LinearSVC operates in a dot-product
# space — ordinal codes would imply false distances between categories).
#
# Numeric features are standardized with StandardScaler inside the CV loop
# to prevent scale leakage across folds.
#
# LabelEncoder is fit once on the full dataset before the CV loop.
# Fitting inside the loop would produce different integer mappings per fold,
# making fold results incomparable and breaking per-drug recall aggregation.

feature_cols     = ['Age', 'Sex', 'Family_income', 'Insurance_coverage',
                    'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
categorical_cols = ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
numeric_cols     = ['Age', 'Family_income', 'Quantity', 'Strength', 'Day_Supply']
target_col       = 'Drug'

le = LabelEncoder()
super_df['Drug_encoded'] = le.fit_transform(super_df[target_col])

print('Unique classes:', len(le.classes_))
print('Feature cols:', feature_cols)
print('Categorical cols:', categorical_cols)
print('Numeric cols:', numeric_cols)
print('\nSample drug to integer mapping (first 5):')
for i, drug in enumerate(le.classes_[:5]):
    print(f'  {drug} -> {i}')

# Hard stop if any defined column is missing from the dataset.
# Catches typos in column names or dataset changes before the CV loop starts.
missing_cols = [c for c in feature_cols + [target_col] if c not in super_df.columns]
assert len(missing_cols) == 0, \
    f"ERROR: These columns are missing from the dataset: {missing_cols}"

# Save the LabelEncoder so it can be reloaded without rerunning this notebook.
# Required for ensemble model — all models must use identical drug-to-integer mappings.
joblib.dump(le, 'svm_super_label_encoder.joblib')
print('\nLabelEncoder saved to svm_super_label_encoder.joblib')

In [ ]:
# Define the preprocessing pipeline.
#
# Numeric pipeline:
#   Step 1: SimpleImputer (median) — fills any remaining demographic NaNs
#           using only train-fold statistics (fitted on train, applied to val).
#           Prescription NaNs were already filled with -1 above, so the imputer
#           will not touch those.
#   Step 2: StandardScaler — standardize to mean=0, std=1.
#           Critical for SVM: the margin is defined in feature space and
#           is sensitive to feature scale. Without scaling, high-range features
#           like Family_income would dominate the decision boundary.
#
# Categorical pipeline:
#   Step 1: SimpleImputer (most_frequent) — fills any remaining categorical NaNs.
#   Step 2: OneHotEncoder — converts categories to binary indicator columns.
#           handle_unknown='ignore' safely handles rare categories in val
#           not seen during training.
#           sparse_output=False returns a dense array compatible with LinearSVC.
#
# NOTE: The preprocessor is fit on train data only inside each fold.
# It is NOT fit here — fitting here would leak val/test statistics into
# imputation medians and scaler means/stds, inflating CV performance.

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(**{OHE_SPARSE_KWARG: False}, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print('Preprocessor defined.')
print('  Numeric pipeline:     SimpleImputer(median) -> StandardScaler')
print('  Categorical pipeline: SimpleImputer(most_frequent) -> OneHotEncoder')
print('  Note: preprocessor is not fitted here — fitted inside each CV fold on train split only.')

In [ ]:
# 5-fold stratified group cross-validation.
#
# StratifiedGroupKFold ensures:
#   1. Each fold has similar drug class distribution (stratified)
#   2. All records for the same person stay in the same fold (grouped)
#      preventing the model from seeing the same person in both train and val
#
# For each fold:
#   1. Split by Person_ID groups and drug label stratification
#   2. Shuffle rows within train and val (prevents clustering of records)
#   3. Fit preprocessor on train split only — apply to both train and val
#   4. Train LinearSVC on preprocessed train fold
#   5. Predict on preprocessed val fold
#   6. Compute and store all metrics + per-drug recall
#
# LinearSVC hyperparameters:
#   C=1.0: regularization strength (higher C = less regularization).
#   class_weight='balanced': scales penalty by inverse class frequency —
#     critical for this severely imbalanced 217-class problem.
#   max_iter=5000: increased from default (1000) to ensure convergence
#     on large data. Warnings about convergence will appear if too low.
#   random_state=42: ensures reproducibility of the solver.
#
# Note on CalibratedClassifierCV in CV loop:
#   We use plain LinearSVC here for CV metrics — predict() is sufficient
#   for accuracy, recall, F1, F2 computation.
#   CalibratedClassifierCV is applied only to the final model in Cell 12
#   where predict_proba() is needed for the ensemble probability output.
#   Wrapping inside the CV loop would significantly increase runtime
#   as calibration itself performs an internal cross-validation.

sgkf = StratifiedGroupKFold(n_splits=5)

X      = super_df[feature_cols].copy()
y      = super_df['Drug_encoded'].values
groups = super_df['Person_ID'].values

fold_results          = []
per_drug_recall_folds = []

for fold_num, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f'\n{"="*50}')
    print(f'FOLD {fold_num}/5')
    print(f'{"="*50}')

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold   = X.iloc[val_idx].copy()
    y_train_fold = y[train_idx]
    y_val_fold   = y[val_idx]

    # Shuffle within each fold to prevent any ordering artifacts.
    # Different seeds for train and val so they are shuffled independently.
    train_order  = np.random.RandomState(42).permutation(len(X_train_fold))
    val_order    = np.random.RandomState(43).permutation(len(X_val_fold))
    X_train_fold = X_train_fold.iloc[train_order].reset_index(drop=True)
    y_train_fold = y_train_fold[train_order]
    X_val_fold   = X_val_fold.iloc[val_order].reset_index(drop=True)
    y_val_fold   = y_val_fold[val_order]

    print(f'Train size: {len(X_train_fold):,} | Val size: {len(X_val_fold):,}')
    print(f'Unique drugs in train: {len(np.unique(y_train_fold))} | val: {len(np.unique(y_val_fold))}')

    # Fit preprocessor on train split only.
    # This ensures median imputation and scaler statistics are computed
    # from train data only — no leakage from val into preprocessing.
    X_train_processed = preprocessor.fit_transform(X_train_fold)
    X_val_processed   = preprocessor.transform(X_val_fold)

    print(f'Preprocessed train shape: {X_train_processed.shape}')

    # Train LinearSVC.
    # class_weight='balanced': automatically weights rare classes higher.
    # max_iter=5000: increased to ensure convergence on large feature space.
    model = LinearSVC(
        C=1.0,
        class_weight='balanced',
        max_iter=5000,
        random_state=42
    )
    model.fit(X_train_processed, y_train_fold)
    print(f'Fold {fold_num} training complete.')

    y_pred_fold = model.predict(X_val_processed)

    # Overall metrics.
    acc   = accuracy_score(y_val_fold, y_pred_fold)
    kappa = cohen_kappa_score(y_val_fold, y_pred_fold)
    mcc   = matthews_corrcoef(y_val_fold, y_pred_fold)

    # Macro and micro averaged metrics via permetrics.
    # Macro: average per class equally — treats rare and common drugs equally.
    # Micro: aggregate all counts — dominated by the most common drugs.
    evaluator = ClassificationMetric(y_val_fold, y_pred_fold)

    macro_precision = evaluator.precision_score(average='macro')
    micro_precision = evaluator.precision_score(average='micro')
    macro_recall    = evaluator.recall_score(average='macro')
    micro_recall    = evaluator.recall_score(average='micro')
    macro_f1        = evaluator.f1_score(average='macro')
    micro_f1        = evaluator.f1_score(average='micro')

    # F2 score weights recall twice as much as precision.
    # Higher recall is more important here because missing a drug class
    # means underestimating its wastewater load.
    macro_f2 = evaluator.fbeta_score(beta=2, average='macro')
    micro_f2 = evaluator.fbeta_score(beta=2, average='micro')

    fold_results.append({
        'fold': fold_num,
        'accuracy': acc,
        'cohen_kappa': kappa,
        'mcc': mcc,
        'macro_precision': macro_precision,
        'micro_precision': micro_precision,
        'macro_recall': macro_recall,
        'micro_recall': micro_recall,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'macro_f2': macro_f2,
        'micro_f2': micro_f2,
    })

    # Per-drug recall for this fold.
    # Used for ensemble model selection — the ensemble picks the best model per drug.
    report = classification_report(
        y_val_fold, y_pred_fold,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_recalls         = {drug: report[drug]['recall'] for drug in le.classes_ if drug in report}
    drug_recalls['fold'] = fold_num
    per_drug_recall_folds.append(drug_recalls)

    print(f'Fold {fold_num} results:')
    print(f'  Accuracy:      {acc:.4f}')
    print(f'  Cohen Kappa:   {kappa:.4f}')
    print(f'  MCC:           {mcc:.4f}')
    print(f'  Macro Recall:  {macro_recall:.4f}')
    print(f'  Micro Recall:  {micro_recall:.4f}')
    print(f'  Macro F2:      {macro_f2:.4f}')

print('\n' + '='*50)
print('ALL FOLDS COMPLETE')
print('='*50)

In [ ]:
# Summarize cross-validation results across all 5 folds.
# Report mean and standard deviation for each metric.
# Standard deviation shows how stable the model is across different data splits.
results_df = pd.DataFrame(fold_results)

# Hard stop if not all 5 folds completed.
# If the CV loop crashed mid-run, this catches it before saving incomplete results.
assert len(results_df) == 5, \
    f"ERROR: Expected 5 fold results, found {len(results_df)} — CV may not have completed."

print('CV RESULTS — MEAN +/- STD ACROSS 5 FOLDS')
print('='*55)
metric_cols = [c for c in results_df.columns if c != 'fold']
for col in metric_cols:
    mean = results_df[col].mean()
    std  = results_df[col].std()
    print(f'  {col:<25}: {mean:.4f} +/- {std:.4f}')

results_df.to_csv('svm_super_cv_results.csv', index=False)
print('\nCV results saved to svm_super_cv_results.csv')
print('All 5 folds confirmed complete.')

In [ ]:
# Compute average per-drug recall across all 5 folds.
# This is the key output for ensemble model selection.
# The ensemble picks whichever base model has the highest recall for each drug.
per_drug_df = pd.DataFrame(per_drug_recall_folds)
drug_cols   = [c for c in per_drug_df.columns if c != 'fold']

mean_drug_recall = per_drug_df[drug_cols].mean().reset_index()
mean_drug_recall.columns = ['Drug', 'Mean_Recall_SVM_Super']
mean_drug_recall = mean_drug_recall.sort_values('Mean_Recall_SVM_Super', ascending=False)

print('Top 20 drugs by mean recall:')
print(mean_drug_recall.head(20).to_string(index=False))

print('\nBottom 20 drugs by mean recall:')
print(mean_drug_recall.tail(20).to_string(index=False))

# Summary stats — how many drugs does SVM recall at all?
# A drug is considered recalled if mean recall > 0 across folds.
# This number goes directly into the paper for model comparison.
drugs_recalled = (mean_drug_recall['Mean_Recall_SVM_Super'] > 0).sum()
print(f'\nDrugs with mean recall > 0:    {drugs_recalled} / {len(mean_drug_recall)}')
print(f'Drugs with mean recall >= 0.1: {(mean_drug_recall["Mean_Recall_SVM_Super"] >= 0.1).sum()}')
print(f'Drugs with mean recall >= 0.5: {(mean_drug_recall["Mean_Recall_SVM_Super"] >= 0.5).sum()}')

mean_drug_recall.to_csv('svm_super_per_drug_recall.csv', index=False)
print('\nPer-drug recall saved to svm_super_per_drug_recall.csv')

In [ ]:
# Train final model on the full training dataset (2014-2021).
# After CV gives confidence in model performance, retrain on all available data
# to maximize signal before evaluating on the held-out 2022 validation set.
#
# The preprocessor is fit on the full training data here (no fold split).
# The fitted preprocessor is reused to transform 2022 data in the next cell.
#
# CalibratedClassifierCV is applied after fitting LinearSVC.
# cv='prefit' tells the calibrator that the model is already trained —
# it applies sigmoid calibration (Platt scaling) without refitting the SVM.
# This converts LinearSVC decision scores into proper probability estimates
# over all 217 classes, required for predict_proba() in the ensemble.

X_final = super_df[feature_cols].copy()
y_final = super_df['Drug_encoded'].values

X_final, y_final = shuffle(X_final, y_final, random_state=42)
X_final = X_final.reset_index(drop=True)

print('Final training data size:', X_final.shape)
print('Number of classes:', len(le.classes_))
print('\nFitting preprocessor on full training data...')

X_final_processed = preprocessor.fit_transform(X_final)
print('Preprocessed shape:', X_final_processed.shape)

# Step 1 — Train LinearSVC on full training data.
print('\nTraining final LinearSVC model...')
svm_model = LinearSVC(
    C=1.0,
    class_weight='balanced',
    max_iter=5000,
    random_state=42
)
svm_model.fit(X_final_processed, y_final)
print('LinearSVC training complete.')

# Step 2 — Wrap with CalibratedClassifierCV to enable predict_proba().
# cv='prefit': model is already fitted — calibration only, no refitting.
# method='sigmoid': Platt scaling, standard choice for SVM calibration.
print('\nCalibrating model for probability outputs...')
final_model = CalibratedClassifierCV(svm_model, cv='prefit', method='sigmoid')
final_model.fit(X_final_processed, y_final)
print('Calibration complete — predict_proba() now available.')

# Save final model and preprocessor.
# Both must be saved together — preprocessor stores training statistics
# needed to correctly transform 2022 data in the next cell.
joblib.dump(final_model, 'svm_super_final_model.joblib')
joblib.dump(preprocessor, 'svm_super_preprocessor.joblib')
print('\nFinal model saved to svm_super_final_model.joblib')
print('Preprocessor saved to svm_super_preprocessor.joblib')

In [ ]:
# Load 2022 super dataset (held-out internal validation set).
# Must use super_data_2022.csv — not data_2022.csv —
# because the model was trained with prescription features.
#
# Preprocessing must mirror training exactly:
#   - Prescription NaNs filled with -1 (same as training)
#   - Preprocessor.transform() (not fit_transform) — uses training statistics
#
# final_model here is CalibratedClassifierCV wrapping LinearSVC.
# predict() returns hard labels (same as plain LinearSVC).
# predict_proba() returns calibrated probability vectors (used for ensemble).

data_2022 = pd.read_csv(f'{DATA_DIR}super_data_2022.csv')
if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

# Same sentinel filling as training — must be identical preprocessing
numeric_prescription_cols = ['Quantity', 'Strength', 'Day_Supply']
data_2022[numeric_prescription_cols] = data_2022[numeric_prescription_cols].fillna(-1)
data_2022['Form'] = data_2022['Form'].fillna('-1')

print('2022 data shape:', data_2022.shape)

# Drop any drugs in 2022 not seen during training
unseen = set(data_2022['Drug'].unique()) - set(le.classes_)
print(f'Unseen drugs in 2022 (will be dropped): {unseen}')
data_2022 = data_2022[data_2022['Drug'].isin(le.classes_)].reset_index(drop=True)
print(f'2022 rows after filtering: {len(data_2022):,}')

X_2022 = data_2022[feature_cols].copy()

# Apply training preprocessor — transform only, do not refit.
# Uses median imputation values and scaler statistics from 2014-2021 training data.
X_2022_processed = preprocessor.transform(X_2022)

y_2022_encoded = le.transform(data_2022['Drug'])
y_pred_2022    = final_model.predict(X_2022_processed)

# Compute validation metrics
acc_2022   = accuracy_score(y_2022_encoded, y_pred_2022)
kappa_2022 = cohen_kappa_score(y_2022_encoded, y_pred_2022)
mcc_2022   = matthews_corrcoef(y_2022_encoded, y_pred_2022)

evaluator_2022    = ClassificationMetric(y_2022_encoded, y_pred_2022)
macro_prec_2022   = evaluator_2022.precision_score(average='macro')
micro_prec_2022   = evaluator_2022.precision_score(average='micro')
macro_recall_2022 = evaluator_2022.recall_score(average='macro')
micro_recall_2022 = evaluator_2022.recall_score(average='micro')
macro_f2_2022     = evaluator_2022.fbeta_score(beta=2, average='macro')
micro_f2_2022     = evaluator_2022.fbeta_score(beta=2, average='micro')

print('\nINTERNAL VALIDATION — MEPS 2022 Results')
print(f'Accuracy:          {acc_2022:.4f}')
print(f'Cohen Kappa:       {kappa_2022:.4f}')
print(f'MCC:               {mcc_2022:.4f}')
print(f'Macro Precision:   {macro_prec_2022:.4f}')
print(f'Micro Precision:   {micro_prec_2022:.4f}')
print(f'Macro Recall:      {macro_recall_2022:.4f}')
print(f'Micro Recall:      {micro_recall_2022:.4f}')
print(f'Macro F2:          {macro_f2_2022:.4f}')
print(f'Micro F2:          {micro_f2_2022:.4f}')

# Save probability outputs for ensemble model.
# Shape: (n_observations, 217) — one row per observation, one column per drug.
# Columns ordered by le.classes_ — identical mapping across all 5 base models.
# predict_proba() works here because final_model is CalibratedClassifierCV —
# plain LinearSVC does not have this method.
proba_2022 = final_model.predict_proba(X_2022_processed)
proba_df   = pd.DataFrame(proba_2022, columns=le.classes_)
proba_df.insert(0, 'Observation_ID', data_2022['Observation_ID'].values)
proba_df.to_csv('svm_super_proba_2022.csv', index=False)
print(f'\nProbability output saved to svm_super_proba_2022.csv')
print(f'Shape: {proba_df.shape} — {len(data_2022):,} observations x 217 drugs + Observation_ID')

In [ ]:
# Save per-drug metrics and overall validation summary for 2022.
report_2022 = classification_report(
    y_2022_encoded, y_pred_2022,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)

drug_recall_2022 = pd.DataFrame([
    {
        'Drug': drug,
        'Recall_2022': report_2022[drug]['recall'],
        'Precision_2022': report_2022[drug]['precision'],
        'F1_2022': report_2022[drug]['f1-score'],
        'Support_2022': report_2022[drug]['support']
    }
    for drug in le.classes_ if drug in report_2022
]).sort_values('Recall_2022', ascending=False)

print('Top 15 drugs by recall on 2022 data:')
print(drug_recall_2022.head(15).to_string(index=False))
print('\nBottom 15 drugs by recall on 2022 data:')
print(drug_recall_2022.tail(15).to_string(index=False))

# Summary stats — mirrors Cell 11 but on held-out 2022 data instead of CV.
# Confirms whether CV recall numbers generalise to unseen data.
drugs_recalled_2022 = (drug_recall_2022['Recall_2022'] > 0).sum()
print(f'\nDrugs recalled on 2022 data (recall > 0): {drugs_recalled_2022} / {len(drug_recall_2022)}')
print(f'Drugs with recall >= 0.1: {(drug_recall_2022["Recall_2022"] >= 0.1).sum()}')
print(f'Drugs with recall >= 0.5: {(drug_recall_2022["Recall_2022"] >= 0.5).sum()}')

drug_recall_2022.to_csv('svm_super_2022_per_drug_metrics.csv', index=False)
print('\nPer-drug 2022 metrics saved to svm_super_2022_per_drug_metrics.csv')

validation_summary = pd.DataFrame([{
    'model': 'SVM_Super',
    'dataset': 'MEPS_2022_internal_validation',
    'accuracy': acc_2022,
    'cohen_kappa': kappa_2022,
    'mcc': mcc_2022,
    'macro_precision': macro_prec_2022,
    'micro_precision': micro_prec_2022,
    'macro_recall': macro_recall_2022,
    'micro_recall': micro_recall_2022,
    'macro_f2': macro_f2_2022,
    'micro_f2': micro_f2_2022,
}])
validation_summary.to_csv('svm_super_validation_summary.csv', index=False)
print('Validation summary saved to svm_super_validation_summary.csv')

# Final output file summary — confirms everything saved correctly.
print('\n' + '='*55)
print('ALL OUTPUTS SAVED')
print('='*55)
print('  svm_super_cv_results.csv')
print('  svm_super_per_drug_recall.csv')
print('  svm_super_2022_per_drug_metrics.csv')
print('  svm_super_validation_summary.csv')
print('  svm_super_proba_2022.csv           <- ensemble input')
print('  svm_super_final_model.joblib       <- CalibratedClassifierCV')
print('  svm_super_preprocessor.joblib')
print('  svm_super_label_encoder.joblib')

## Output Files

| File | Contents |
|------|----------|
| `svm_super_cv_results.csv` | Mean +/- std for all metrics across 5 CV folds |
| `svm_super_per_drug_recall.csv` | Average recall per drug across 5 folds (for ensemble selection) |
| `svm_super_2022_per_drug_metrics.csv` | Per-drug recall, precision, F1 on MEPS 2022 internal validation |
| `svm_super_validation_summary.csv` | Overall validation metrics on MEPS 2022 |

**Next step:** Compare these results against the base SVM (demographics only) and against other super dataset models (XGBoost, RealMLP, KNN) to assess whether the prescription features improve SVM performance.  
The per-drug recall CSV feeds directly into Vanessa's ensemble selection logic.